In [3]:
import pandas as pd
import pickle

# Load train_dataset 
train_data = pd.read_csv('datasets/train_dataset.csv')

# Separate features and target
X = train_data.drop(['isFraud'], axis=1)
y = train_data['isFraud']

# Load the 10-fold indices
with open("10_fold_indices.pkl", "rb") as f:
    splits = pickle.load(f)

print(f"Loaded {len(splits)} folds.")

Loaded 10 folds.


In [4]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
import warnings
warnings.filterwarnings("ignore")

# -------------------- Setup --------------------
os.makedirs("models/logistic_regression_te", exist_ok=True)

auc_scores = []
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# -------------------- Column Types --------------------
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# -------------------- Stratified K-Fold --------------------
for fold, (train_idx, val_idx) in enumerate(splits, 1):
    print(f"\nFold {fold}")

    # Split fold data
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Target Encode
    encoder = TargetEncoder(cols=categorical_cols)
    X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols], y_train)
    X_val[categorical_cols] = encoder.transform(X_val[categorical_cols])

    # Standardize numeric features
    scaler = StandardScaler()
    X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
    X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])

    # Initialize model
    model = LogisticRegression(
        solver="saga",
        max_iter=1,
        warm_start=True,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    best_auc = 0
    best_model = None
    rounds_without_improvement = 0
    max_epochs = 5000
    patience = 50

    for epoch in range(1, max_epochs + 1):
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_val_prob)

        if auc > best_auc:
            best_auc = auc
            best_model = pickle.loads(pickle.dumps(model))  # Deep copy
            rounds_without_improvement = 0
        else:
            rounds_without_improvement += 1

        if epoch % 50 == 0 or rounds_without_improvement == patience:
            print(f"Epoch {epoch} AUC-ROC: {auc:.4f}")

        if rounds_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch} | Best AUC-ROC: {best_auc:.4f}")
            break

    # Final evaluation
    y_val_pred = best_model.predict(X_val)
    y_val_prob = best_model.predict_proba(X_val)[:, 1]

    auc = roc_auc_score(y_val, y_val_prob)
    acc = accuracy_score(y_val, y_val_pred)
    prec = precision_score(y_val, y_val_pred, zero_division=0)
    rec = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)

    auc_scores.append(auc)
    accuracy_scores.append(acc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    f1_scores.append(f1)

    print(f"Fold {fold} Results:")
    print(f"  AUC-ROC : {auc:.4f}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1 Score : {f1:.4f}")

    # Save model
    with open(f"models/logistic_regression_te/fold_{fold}.pkl", "wb") as f:
        pickle.dump(best_model, f)

# -------------------- Final Summary --------------------
print("\nCross-Validation Results (Logistic Regression + Target Encoding)")
for i in range(len(auc_scores)):
    print(f"Fold {i+1}: AUC={auc_scores[i]:.4f} | Acc={accuracy_scores[i]:.4f} | Prec={precision_scores[i]:.4f} | Recall={recall_scores[i]:.4f} | F1={f1_scores[i]:.4f}")

print("\nFinal Averages:")
print(f"AUC-ROC : {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Precision: {np.mean(precision_scores):.4f}")
print(f"Recall   : {np.mean(recall_scores):.4f}")
print(f"F1 Score : {np.mean(f1_scores):.4f}")



Fold 1
Epoch 50 AUC-ROC: 0.8420
Epoch 100 AUC-ROC: 0.8442
Epoch 150 AUC-ROC: 0.8453
Epoch 200 AUC-ROC: 0.8462
Epoch 250 AUC-ROC: 0.8469
Epoch 300 AUC-ROC: 0.8474
Epoch 350 AUC-ROC: 0.8479
Epoch 400 AUC-ROC: 0.8483
Epoch 450 AUC-ROC: 0.8486
Epoch 500 AUC-ROC: 0.8489
Epoch 550 AUC-ROC: 0.8492
Epoch 600 AUC-ROC: 0.8495
Epoch 650 AUC-ROC: 0.8498
Epoch 700 AUC-ROC: 0.8500
Epoch 750 AUC-ROC: 0.8502
Epoch 800 AUC-ROC: 0.8504
Epoch 850 AUC-ROC: 0.8507
Epoch 900 AUC-ROC: 0.8509
Epoch 950 AUC-ROC: 0.8510
Epoch 1000 AUC-ROC: 0.8512
Epoch 1050 AUC-ROC: 0.8514
Epoch 1100 AUC-ROC: 0.8516
Epoch 1150 AUC-ROC: 0.8518
Epoch 1200 AUC-ROC: 0.8519
Epoch 1250 AUC-ROC: 0.8521
Epoch 1300 AUC-ROC: 0.8522
Epoch 1350 AUC-ROC: 0.8524
Epoch 1400 AUC-ROC: 0.8525
Epoch 1450 AUC-ROC: 0.8527
Epoch 1500 AUC-ROC: 0.8528
Epoch 1550 AUC-ROC: 0.8529
Epoch 1600 AUC-ROC: 0.8531
Epoch 1650 AUC-ROC: 0.8532
Epoch 1700 AUC-ROC: 0.8533
Epoch 1750 AUC-ROC: 0.8534
Epoch 1800 AUC-ROC: 0.8535
Epoch 1850 AUC-ROC: 0.8537
Epoch 1900 AU